# RailGuard Vision — Phase 1: Dataset Setup & Preparation (Colab)

**Project:** RailGuard Vision — YOLOv8 Railway Track Defect Detection System  
**Objective:** Download, clean, inspect, stratify, annotate, and augment railway track fault datasets for YOLOv8 object detection model training.

---

## Step 1: Install & Configure Kaggle API and Download Dataset

In [ ]:
# Install Kaggle API client
!pip install -q kaggle albumentations opencv-python matplotlib pandas scikit-learn pillow

import os
from google.colab import files

# Upload your kaggle.json API key file if running in Google Colab
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Please upload your kaggle.json API token:')
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download Railway Track Fault Detection Dataset from Kaggle
!kaggle datasets download -d akshayartani/railway-track-fault-detection -p ./data/raw --unzip
print('Dataset download and unzipping complete!')

## Step 2: Inspect Folder Structure & Class Distribution

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

raw_dir = Path('./data/raw')
class_counts = {}

for item in raw_dir.glob('*'):
    if item.is_dir():
        images = list(item.glob('*.jpg')) + list(item.glob('*.png')) + list(item.glob('*.jpeg'))
        class_counts[item.name] = len(images)

print('--- Dataset Class Distribution ---')
for cls_name, count in class_counts.items():
    print(f'Class: {cls_name:<20} | Image Count: {count}')

# Plot Class Distribution
plt.figure(figsize=(8, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='#2B6CB0')
plt.title('Railway Fault Dataset Class Distribution')
plt.xlabel('Defect Class')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Step 3: Dataset Cleaning (Remove Corrupted Files, Exact Duplicates & Convert to JPEG)

In [ ]:
import hashlib
from PIL import Image
import cv2

def file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

cleaned_dir = Path('./data/cleaned')
cleaned_dir.mkdir(parents=True, exist_ok=True)

seen_hashes = set()
corrupted_count = 0
duplicate_count = 0
valid_count = 0

for cls_folder in raw_dir.glob('*'):
    if not cls_folder.is_dir():
        continue
    out_cls_dir = cleaned_dir / cls_folder.name
    out_cls_dir.mkdir(parents=True, exist_ok=True)
    
    for img_path in cls_folder.glob('*.*'):
        # 1. Check corrupt file
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception:
            corrupted_count += 1
            continue
        
        # 2. Check exact duplicates via MD5 Hash
        h = file_hash(img_path)
        if h in seen_hashes:
            duplicate_count += 1
            continue
        seen_hashes.add(h)
        
        # 3. Read & standardize format to JPEG (RGB)
        cv_img = cv2.imread(str(img_path))
        if cv_img is None:
            corrupted_count += 1
            continue
        
        save_path = out_cls_dir / f'{img_path.stem}.jpg'
        cv2.imwrite(str(save_path), cv_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        valid_count += 1

print(f'Cleaning Complete: {valid_count} Valid, {duplicate_count} Duplicates Removed, {corrupted_count} Corrupted Files Purged.')

## Step 4: Stratified Train / Val / Test Split (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

split_base = Path('./data/dataset_split')
for split in ['train', 'val', 'test']:
    (split_base / split / 'images').mkdir(parents=True, exist_ok=True)
    (split_base / split / 'labels').mkdir(parents=True, exist_ok=True)

all_files = []
all_labels = []

for cls_dir in cleaned_dir.glob('*'):
    if cls_dir.is_dir():
        for img in cls_dir.glob('*.jpg'):
            all_files.append(img)
            all_labels.append(cls_dir.name)

# 70% train, 30% temp
train_f, temp_f, train_l, temp_l = train_test_split(all_files, all_labels, test_size=0.30, stratify=all_labels, random_state=42)
# Split 30% temp equally into 15% val and 15% test
val_f, test_f, val_l, test_l = train_test_split(temp_f, temp_l, test_size=0.50, stratify=temp_l, random_state=42)

print(f'Train samples: {len(train_f)} | Val samples: {len(val_f)} | Test samples: {len(test_f)}')

def copy_split_files(file_list, split_name):
    dest_dir = split_base / split_name / 'images'
    for f in file_list:
        shutil.copy(f, dest_dir / f.name)

copy_split_files(train_f, 'train')
copy_split_files(val_f, 'val')
copy_split_files(test_f, 'test')
print('Dataset stratified splitting finished!')

## Step 5: Bounding Box Annotation (Roboflow Workflow) & `data.yaml` Generation

### Roboflow Bounding Box Annotation Workflow:
If the Kaggle dataset only provides classification sub-folders (e.g. `Defective`, `Non-defective`), object detection requires bounding boxes `[x_center, y_center, width, height]`.
1. Export the split image folders (`./data/dataset_split`) and upload to **Roboflow.com**.
2. Draw tight bounding boxes around rail defects (cracks, missing fasteners, joint faults, broken rails).
3. Export the dataset in **YOLOv8 PyTorch format**.
4. Download and unzip into `./data/yolo_dataset` containing `data.yaml`.

In [ ]:
# Generate standard placeholder data.yaml for YOLOv8
data_yaml_content = """path: ./data/dataset_split
train: train/images
val: val/images
test: test/images

names:
  0: crack
  1: missing_fastener
  2: broken_rail
  3: joint_fault
"""

with open('./data/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print('Generated YOLO data.yaml configuration file at ./data/data.yaml')

## Step 6: Albumentations Data Augmentation Pipeline & Visualization

In [ ]:
import albumentations as A
import cv2
import matplotlib.pyplot as plt

# Define Albumentations augmentation pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.8),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3)
])

# Load sample image for demonstration
sample_path = list((split_base / 'train' / 'images').glob('*.jpg'))[0]
sample_image = cv2.imread(str(sample_path))
sample_image = cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB)

augmented = transform(image=sample_image)['image']

# Plot Before and After Augmentation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(sample_image)
ax1.set_title('Original Track Image')
ax1.axis('off')

ax2.imshow(augmented)
ax2.set_title('Augmented Image (Flip + CLAHE + Contrast)')
ax2.axis('off')

plt.tight_layout()
plt.show()
print('Phase 1 Dataset Setup and Augmentation Completed!')